#  2.2 Extracting information from literature

In many scientific fields, including biology, a large portion of knowledge is published in unstructured text such as research articles, reviews, and supplementary materials. Important information about genes, proteins, experimental methods, and biological interactions is often embedded within this text rather than stored in structured databases. 

As the volume of scientific literature continues to grow rapidly, manually extracting and organizing this information becomes increasingly difficult and time-consuming. Automating data extraction from the literature using AI can help researchers rapidly identify relevant findings, convert unstructured text into structured datasets, and integrate knowledge across thousands of publications. 

# 2.2.1 Getting started with LLMs

**What are APIs?**

API (Application Programming Interface): This is the mechanism that enable seamless communication and data exchange between different software systems. Eg: OpenAI model and your request. Defines the available functions, data formats, and protocols for interaction.

API Key: A unique string to authenticate the user. In this example let's use a GPT model. For this, you need an API key from OpenAI. You can get your key from [OpenAI platform.](https://platform.openai.com/api-keys)

*Note:* You can may simply swap the language model in the following example if you wish to use a different model/provider.

**1) Set your API key for this notebook**
Let's start by opening this notebook as a `Google Colab notebook`.

- Once you have received a unique API key from OpenAI. Setup the API keys by clicking key icon on on the left tab. More info [here](https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb)

**2) Install the requirements to run examples in this notebook.**


In [ ]:
!pip install openai pandas matplotlib requests datasets trl peft nltk rank_bm25

You can use the code snippet below to see if you have set the key correctly.

In [ ]:
import os
from google.colab import userdata

# Access the OpenAI API key
openai_api_key = userdata.get('OPENAI_API_KEY')

# Use the API key for API requests
print(f"Your OpenAI API Key: {openai_api_key}")

## Let's test a basic API request

In [ ]:
import openai
from openai import OpenAI
import os
from google.colab import userdata

# Access the OpenAI API key
openai_api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=openai_api_key)

response = client.responses.create(
    model="gpt-4.1-mini",
    input="What is the difference between Machine Learning and Deep Learning?"
)

print(response.output_text)

# <font color="blue"> How is scicence advancing with the recent advances in LLMs/AI? </font>

A statement that's everywhere now: "The recent advances of LLMs are accelerating scientific research and discovery".

But why is that? Natural language is like a thread which binds all sorts of information, data, software and applications. For exampls, natural language is what we use to extract information from literature, access databases and even write python code.

That is why LLMs have been an integral component in automating scientific discovery. For example LLMs can be used to read papers and extract their knowledge at scale or access internet to search for an answer. These LLMs with human-like performance allow us to connect many isolated tools in science and adapt them for our domain specific applications.

# <font color="blue"> Another important question to ask is, can LLMs alone be useful? </font>

Yes and No -- it depends on the task. While current state-of-the-art LLMs have been trained on "all possible data" they still have many limitations. LLMs can alone be used to do certain tasks such as writing essays, summarize text etc.

But it might fail to answer domain specific more nuanced questions without additional information. eg: How many hours did X et al. spent during their experiment of Y?

Most times LLMs would try to provide some answer confidentally although they're wrong. These are called **hallucinations**.

You can define hallucinations as incorrect and misleading results that AI models generate.

Commonly approaches to reduce hallucinations:
- consensus sampling: generate answers multiple times by a model and accept the answer with highest mode.
- using verification methods: You can use a python function to verify the generate answer
- integrate additional capabilities (tools) to LLMs: an LLM that can browse the internet.

**References**
- [Exploring the role of large language models in the scientific method: from hypothesis to discovery](https://www.nature.com/articles/s44387-025-00019-5#:~:text=Large%20language%20models%20(LLMs)%20can%20be%20used,good%20observers%20for%20language%20and%20image%20data)
- [The future of chemistry is language](https://www.nature.com/articles/s41570-023-00502-0)

# <font color="Purple"> LLMs + Tools = Agents </font>

This is where AI agents come into play. In more detail, the modern idea of **AI agents** connect closely with reinforcement learning (RL) but it actually runs deeper than that. Read more on *cybernetics* and *classical AI* if you're interested.

However, RL provide the mathematical formulation to the AI agents that we will be using.

## Few definitions

**Agents:**

In simplest terms an AI Agent is an LLM with externally incorporated tools. But by definition, an agent is an entity that interacts with an **environment**, perceives **states**, chooses **actions**, and receives **rewards**. This flow can be described with a **Markov Decision Process (MDP)**. This also means that the current state of an agent only depends on the state before that.

$s_t \rightarrow A_t \rightarrow R_{t+1},S_{t+1}$

In other words, a single cell can be thought of as an agent: it senses chemicals around it (input), decides to move or release signals (action), and this all happens in its environment. That’s basically an agent acting in an environment!

There are different "types" of AI agents, eg: Simple Agents, ReAct Agent. Read more [here](https://www.notion.so/futurehouse/Simple-and-ReAct-24f3f583b6908087a8d7e3a79900da3c?source=copy_link)


**Environments:**

The agent observes or senses the environment — like how a biologist might observe cell behavior under a microscope. The environment is everything outside the agent that it can interact with. This could be:
- A virtual environment (like a game world or a simulation)

- The real world (a robot moving around a lab)

- A dataset or a web API

**States:**

The state is a snapshot of the agent’s current situation. ie. what the agent knows. It includes everything relevant to making a decision. For a cell agent, this might be the concentration of nutrients nearby. For a software agent, it might be data it has gathered. You can think of the state as the agent’s **memory or current understanding** of the environment.

**Tools**

Agents use tools to improve its performance. Think of tools as **resources or specialized skills** the agent can call on. For example, an AI agent answering biology questions might use a database of research papers as a tool. A robot might have a microscope or a pipette as physical tools. In software, tools might be APIs, calculators, or data analysis modules.


**A few examples of how to use agents in research:**

- Literature search

- Automating data analysis

- Hypothesis generation

- Evaluating models,agents

- Write technical reports

## **Q&A Break !**

#  <font color="blue"> Let's create an agent to extract information from literature! </font>

This type of agents/workflows are now called Retrieval-Augmented Generation (RAG) ([ref](https://arxiv.org/abs/2005.11401)). It is an AI framework that enhances LLM accuracy by fetching data from external, trusted sources before generating an answer. It reduces hallucinations and ensures up-to-date information by combining retrieval (finding relevant data) with generation (producing answers).

Key steps:
- Retrieving relevant documents
- Using a language model to generate insights
- Post-processing the results


## <font color="purple"> Step 1: Knowledge Base Construction </font>

Firstly, we need access to a knowledge base. This could be a locally existing folder of PDF files or an online database. If you're building your own database, you have to use **a storage** (a vector database such as FAISS) where you will store vectorized documents and **a search algorithm**. Commonly we use cosine similatity between the query and documents to find "most relevant" documents to answer the query.


In this example, we'll use the PubMed API for retrieving scientific papers. To keep this simple and "free", we will only be using the abstracts of the most relevant papers in PubMed. With these the database and the search algorithm are taken care through PubMed database API.

Take a look at the following two functions,

1. `fetch_pubmed_ids`: Retrieve PubMed IDs for papers matching a query
    - This function sends a request to PubMed’s API with your query and asks for up to `max_results` article IDs (unique identifiers) that match the query. Then it returns a list of PubMed article IDs that match your search.

2. `fetch_pubmed_abstracts`: Fetch article abstracts from PubMed, given article IDs.
    - This function sends a request to PubMed’s API to get detailed information about each article and extracts the abstract text. Then it returns a list of abstracts for the articles.

In [ ]:
import requests
import xml.etree.ElementTree as ET


# === Step 1: Fetch PubMed article IDs for a query ===
def fetch_pubmed_ids(query:str, max_results:int) -> list[str]:
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "xml"
    }
    resp = requests.get(url, params=params)
    root = ET.fromstring(resp.content)
    ids = [id_elem.text for id_elem in root.findall(".//IdList/Id")]
    return ids


# === Step 2: Fetch abstracts for given PubMed IDs ===
def fetch_pubmed_abstracts(id_list: list[str]) -> list:
    if not id_list:
        return []
    ids_str = ",".join(id_list)
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed",
        "id": ids_str,
        "retmode": "xml"
    }
    resp = requests.get(url, params=params)
    root = ET.fromstring(resp.content)
    abstracts = []
    for article in root.findall(".//PubmedArticle"):
        abstract_texts = article.findall(".//AbstractText")
        # Join multi-part abstracts if needed
        abstract = " ".join([elem.text for elem in abstract_texts if elem.text])
        abstracts.append(abstract)
    return abstracts

## <font color="purple"> Step 2: Retrieve the most relevant abstracts </font>

We should not use all "hits" we get from PubMed. Instead, we should first identify the *most relevant* documents. There are many ways of doing this, but in this tutorial we'll use [BM25](https://en.wikipedia.org/wiki/Okapi_BM25) (a traditional information retrieval method) to fetch the most relevant documents from the knowledge base. You can read more on RAG [here](https://cloud.google.com/use-cases/retrieval-augmented-generation?hl=en).

We will use the following `retrieve_relevant_abstracts` function to rerank the abstract we received from PubMed.

- `retrieve_relevant_abstracts`: This function finds the most relevant abstracts from the list, based on how well they match your query.

    It breaks each abstract into words (tokens) -> Then it uses the BM25 algorithm to rank documents by relevance to your query -> Finally, it selects the top top_k most relevant abstracts.

In [ ]:
import nltk
nltk.download('punkt_tab')
from rank_bm25 import BM25Okapi

# === BM25 Retrieval to find most relevant abstracts ===
def retrieve_relevant_abstracts(abstracts: list[str], query: str, top_k: int) -> list[str]:
    tokenized_corpus = [nltk.word_tokenize(doc.lower()) for doc in abstracts]
    bm25 = BM25Okapi(tokenized_corpus)
    tokenized_query = nltk.word_tokenize(query.lower())
    scores = bm25.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [abstracts[i] for i in top_indices]

## <font color="purple"> Step 3: Answer Generation </font>

Now that we have the retrieval step down, we can use an LLM to augment the generated text. In this example we'll use a `gpt-4.1-mini` (or your favorite GPT model).

*<font color="red">Again, make sure you have set the API key using the key icon on the left side tab.</font>*

We will show the query and the most relevant abstracts to our LLM and ask it to generate the answer.

You can also expand this workflow to include multimodal capabilities in your RAG pipeline so it can even work with plots and figures.

- `generate_answer`: This function uses an LLM model (`gpt-4.1-mini`) to generate a detailed answer or explanation based on the relevant abstracts and your query.

- Here, the retireved abstracts (chunks) are combined into one text block (context), which is sent to the LLM. The LLM then reads all the information and generates an informative answer.

- You can edit the prompt given here and see how it affects the model output.

In [ ]:
import os
import openai
from openai import OpenAI
from google.colab import userdata

#set openai api key
openai_api_key = userdata.get('OPENAI_API_KEY')

client = OpenAI(api_key=openai_api_key)

PROMPT = """Answer the question based on the context provided.

Context:
{context_text}


Question:
{query}

Answer: """

# === Use OpenAI LLM to generate answer given retrieved abstracts and query ===
def generate_answer_with_openai(contexts: list[str], query: str, llm_model: str="gpt-4") -> str:
    context_text = "\n\n".join(contexts)
    prompt = PROMPT.format(context_text=context_text, query=query)
    messages = [
        {"role": "system", "content": "You are a helpful scientific assistant. "},
        {"role": "user", "content": prompt}
    ]

    response = client.responses.create(
        model=llm_model,
        input=messages,
    )

    answer = response.output_text
    return answer

## <font color="blue"> RAG workflow: put the pieces together </font>

This is the main function that ties everything together and runs the full Retrieval-Augmented Generation (RAG) pipeline. Given your query, it will

1. Search PubMed for relevant article IDs
2. Fetch the abstracts for those articles
3. Select the most relevant abstracts using BM25
4. Feed the relevant abstracts and your query into the specified LLM model
5. Generate a detailed answer

In [ ]:
PUBMED_MAX_RESULTS = 10 #number of articles to fetch from pubmed
TOP_K = 5 #number of most relevant abstracts to retrieve
LLM_MODEL = "gpt-4.1-mini" #model to use for text generation

def rag_pipeline(query: str, max_results: int = PUBMED_MAX_RESULTS, top_k: int = TOP_K, llm_model: str = LLM_MODEL) -> str:
    #query = "kinase inhibitors for cancer treatment"
    pubmed_ids = fetch_pubmed_ids(query, max_results)
    if not pubmed_ids:
        print("No PubMed articles found for this query.")
        return None


    print(f"Fetched {len(pubmed_ids)} PubMed article IDs.")

    abstracts = fetch_pubmed_abstracts(pubmed_ids)
    print(f"Retrieved {len(abstracts)} abstracts.")

    print("Retrieving most relevant abstracts using BM25...")
    relevant_abstracts = retrieve_relevant_abstracts(abstracts, query, top_k)
    for i, abs_text in enumerate(relevant_abstracts, 1):
        print(f"\n--- Abstract {i} ---\n{abs_text[:500]}...")  # print first 500 chars

    print("\nGenerating answer using OpenAI GPT-4...\n")
    answer = generate_answer_with_openai(relevant_abstracts, query, llm_model)
    print("=== Generated Answer ===")
    print(answer)

    return answer

In [ ]:
# To run this pipeline, you can simply do

# Since we use pubmed the questions will have to something "biology"
user_query = "kinase inhibitors for cancer treatment"
answer = rag_pipeline(user_query)

## <font color="purple"> A platform for using advaned agents </font>

Edison Scientific (for profit spin off from FutureHouse) provides [a commercial platform](https://platform.edisonscientific.com/) to do research at scale. These are more advanced and engineered agents compared to what we just built.

You can create your own account with free credits with your `.edu` email. Feel free to checkout the Literature agent.

You may already understand now, just as we can use single agents, we can combine multiple agents to create a multi-agents system. Kosmos in Edison Platform is one such example.

## <font color="purple"> Standard approaches to deploying agents </font>

Anthtopic has released an open-source protocol known as Model Context Protocol (MCP) to standardize the these practices. Read more [here](https://www.anthropic.com/news/model-context-protocol).

MCP can be used to connect "AI assistants to the systems where data lives, including content repositories, business tools, and development environments." There are many off-the-shelf tools and code snippets to build your own agents.

Here are some useful resources:

- [what is an MCP?](https://www.youtube.com/watch?v=eur8dUO9mvE)
- [MCP vs API](https://www.youtube.com/watch?v=7j1t3UZA1TY)
- [Concepts of MCP](https://modelcontextprotocol.io/docs/learn/architecture)

## <font color="blue"> AI Safety </font>

AI safety is about ensuring that AI systems:

- Do what we intend them to do
- Do not cause harm (to individuals, society, or science)
- Remain reliable and controllable as they become more capable

AI systems are increasingly used in high-stakes domains (medicine, science, hiring, policy), used autonomously or semi-autonomously. This creates many  risks such as:

- Incorrect but confident outputs (hallucinations)
- Bias and unfair outputs
- Misuse or unintended consequences

Therefore, we have to ensure that we take safety measures during and post training. The commercial LLMs usually go through alignment and red teaming efforts before they release their models. But still it is not uncommon to jailbreak these models.

References:
- https://ai.google/safety/
- https://safe.ai/
- https://www.aisi.gov.uk/

**Why this matters for scientists?**

Reproducibility and trusworthiness in data is cruitial in science. For example, in most cases we work with sensitive data or proprietory. eg: virus data, business information. Or in other cases errors can propagate into publications. Therefore, using AI responsibly is part of scientific rigor.

# Q&A

I'd like to answer any questions you have (related or unrelated to the material we discussed in class today)

## Discussion points

If you'd like we can discuss a few topics based on what we saw today...

- Evaluating LLMs vs Evaluating Agents
- AI safety, what does this mean to you and why do you think this is important?
- Are agents the best way to accelerate scientific discovery?
- As a scientist how do you feel about integrating AI with your research?
- What are some big failures you have observed with LLMs and agents?
- Open-source models: pros and cons
- Any interesting project ideas?
- Many new startups and FROs are focusing on AI x biology domain


Contact info:
geemi@futurehouse.org